# Analisis exploratorio - Safety Save (MID1)

Antes de vectorizar con TF-IDF, revisamos dos patrones:
1. Distribucion de riesgo (alto/bajo) por contratista
2. Palabras mas frecuentes en cada categoria de riesgo

Asegurate de subir `safety_save_limpio.csv` a esta misma carpeta de Colab (o montar Drive) antes de correr la celda.


In [ ]:
import pandas as pd
import re
from collections import Counter

# Cargar el dataset limpio
df = pd.read_csv('safety_save_limpio.csv')

# --- Patron 1: distribucion de riesgo por contratista ---
pivot = df.groupby(['contratista', 'riesgo']).size().unstack(fill_value=0)
pivot = pivot.reindex(columns=['alto', 'bajo'], fill_value=0)
pivot['total'] = pivot['alto'] + pivot['bajo']
pivot = pivot.sort_values('alto', ascending=False)
print(pivot)

# --- Patron 2: palabras mas frecuentes por nivel de riesgo ---
def tokenize(text):
    # Extrae palabras de 3+ letras, todo en minusculas
    return re.findall(r'\b[a-zA-Z]{3,}\b', text.lower())

# Conectores que no aportan senal (nota: dejamos "not" a proposito,
# porque niega la accion y es relevante en este dominio)
skip = {'the', 'and', 'for', 'are', 'were'}

alto_words = Counter()
bajo_words = Counter()

for _, row in df.iterrows():
    tokens = [w for w in tokenize(row['texto']) if w not in skip]
    if row['riesgo'] == 'alto':
        alto_words.update(tokens)
    else:
        bajo_words.update(tokens)

print("\nTop 8 palabras en observaciones de riesgo ALTO:")
for palabra, cuenta in alto_words.most_common(8):
    print(f"  {palabra}: {cuenta}")

print("\nTop 8 palabras en observaciones de riesgo BAJO:")
for palabra, cuenta in bajo_words.most_common(8):
    print(f"  {palabra}: {cuenta}")
